In [ ]:
from arch.univariate import GARCH
import numpy as np

In [ ]:
def get_sigmas_from_garch_params(sigma_start, omega, alpha, beta, n_paths, n_steps):
    garch = GARCH(p=1, q=1)
    errors = np.zeros((n_paths, n_steps))
    variances = np.zeros((n_paths, n_steps))
    for i in range(n_paths):
        output = garch.simulate(
            parameters=[omega, alpha, beta],
            nobs=n_steps,
            initial_value=sigma_start**2,  # starting sigma² (variance)
            rng=np.random.standard_normal,
            burn=1
        )
        errors[i]=output[0]
        variances[i]=output[1]
    return (errors, variances)

In [70]:
omega = 1
alpha = 0.5
beta = 0.01
n_paths=2
n_steps=5
sigma_start=1

get_sigmas_from_garch_params(sigma_start, omega, alpha, beta, n_paths, n_steps)

array([[-1.35805993, -0.71836682,  0.4422259 , -0.7115401 ,  1.76872753],
       [ 3.19980421, -2.90335379,  6.37087166,  4.86775246,  1.51996645]])

In [ ]:
### Write a function that does a true Monte-Carlo Simulation of the Black-Scholes Call Option price 
#with Delta based control variants

def bs_MC_call(S0, K, sigma, r, t, mu = 0, n_sims = 2500, n_hedges = 50, delta_sims = 250):
    
    """Description
    Monte-Carlo simulation of the Black-Scholes value of a call option with Delta based control variants
    
    
    Parameters:
    S0 (float): spot price
    K (float): strike price
    sigma (float): volatility
    r (float): risk-free interest rate
    t (float): time to expiration
    mu (float): Drift of log-returns
    n_sims (int): Number of simulations
    n_hedges (int): number of delta control variants at evenly spaced increments
    
    
    Return:
    np.array of simulated values of Black-Scholes value of call option
    """

    #Noise in volatility
    noise = np.random.normal(0,1,size = (n_sims, n_steps))
    
    #Custom sigma that is not constant

    sigma = np.zeros((n_sims, n_steps))

    for i in range(n_sims):
        simulation = model.simulate(res.params, nobs=n_steps)
        sigma[i] = simulation['errors'].to_numpy()/100
        
    #Time increment between each step
    dt = 1
    
    #log-returns between each step
    increments = (res.params.mu/100 + r*(1/252) - .5*sigma**2) + sigma*np.sqrt(dt)*noise
    
    #Cumulative log-returns at each step
    log_returns = np.cumsum(increments, axis = 1)
    
    
    #paths
    paths = S0*np.exp(log_returns)



    #Simulate call payouts discounted to time 0

    path_end_points = paths[:,-1]

    call_payouts = np.maximum(path_end_points - K,0)*np.exp(-r*t)



    #Simulate stock profits at each interval

    ## profit from start to first step discounted to time 0
    ### We are going to cheat at the current moment in our simulation
    ### We will use the Black-Scholes formula to find Delta,
    ### We'll simulate Delta later
    

    delta_start = monte_sim_call_delta(S0,K,t,r,delta_sims)

    paths_first_steps = paths[:,0]

    first_stock_profits = (paths_first_steps - S0*np.exp(dt*r))*np.exp(-dt*r)

    stock_profits = []

    stock_profits.append(first_stock_profits)




    ## stock profits in intermediate steps

    for i in range(1,n_hedges):
        stock_start = paths[:,i-1]
        stock_end = paths[:,i]
        tte = t-i*dt
        deltas = bs_MC_call_delta_array(stock_start, K, sigma, tte, r,delta_sims)


        stock_profit = (stock_end - stock_start*np.exp(r*dt))*deltas*np.exp(-i*dt*r)


        stock_profits.append(stock_profit)


    total_stock_profit = np.sum(stock_profits, axis = 0)

    profits_hedged = call_payouts - total_stock_profit
    
    
    return profits_hedged
